In [2]:
import torch
torch.cuda.is_available()
!nvidia-smi

Tue Dec  2 23:15:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from __future__ import annotations
from dataclasses import dataclass, field, asdict
from typing import Callable, Dict, List, Tuple, Optional, Any
import json, math, re, textwrap, random, os, sys
from collections import Counter, defaultdict

def normalize_entry(entry: Dict[str, Any]) -> Dict[str, Any]:
    attr = entry.get("attributes", {})

    name = attr.get("animalName", "")
    species = attr.get("animalSpecies", "")
    breed = attr.get("animalBreed", "")
    age = attr.get("animalAgeString", "")
    sex = attr.get("animalSex", "")
    desc = attr.get("animalDescriptionPlain", "")
    location = attr.get("animalLocation", "")

    # title for ranking
    title = f"{name} - {species}"

    # text blob used for TF-IDF
    text = (
        f"name {name} "
        f"species {species} "
        f"breed {breed} "
        f"age {age} "
        f"sex {sex} "
        f"location {location} "
        f"description {desc}"
    )

    normalized = {
        "id": entry.get("id"),
        "name": name,
        "species": species,
        "breed": breed,
        "age": age,
        "sex": sex,
        "location": location,
        "description": desc,
        "pictures": attr.get("animalPictures", []),

        # internal search fields
        "title": title,
        "text": text,
    }

    return normalized

with open("/content/toy_corpus.json", "r") as f:
    raw_json = json.load(f)
raw_list = raw_json["data"]

# Normalize all animals
CORPUS = [normalize_entry(e) for e in raw_list]

# Tokenize the document into words
def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9']+", text.lower())

# Get all the words of each document in the corpus
DOC_TOKENS = [tokenize(d["title"] + " " + d["text"]) for d in CORPUS]

# Get all the words from the corpus
VOCAB = sorted(set(t for doc in DOC_TOKENS for t in doc))

# Compute term frequency (TF) for each doc
def compute_tf(tokens: List[str]) -> Dict[str, float]:

    counts = defaultdict(int)
    for token in tokens:
        counts[token] += 1

    length = max(1, len(tokens))

    return {token: counts[token] / length for token in counts}

# Compute the document frequency across corpus: how many docs does a word appear?
def compute_df(doc_tokens: List[List[str]]) -> Dict[str, float]:
    df = defaultdict(int)
    for tokens in doc_tokens:
        for token in set(tokens):
            df[token] += 1
    return df

# Compute the inverse document frequency (higher for rarer terms), in which we use a smoothed variant
DF = compute_df(DOC_TOKENS)
N_DOC = len(DOC_TOKENS)
IDF = {t: math.log((N_DOC + 1) / (DF[t] + 0.5)) + 1 for t in VOCAB}


# Compute TF-IDF vectors for each document, which is the product between
def tfidf_vector(tokens: List[str]) -> Dict[str, float]:
    tf = compute_tf(tokens)
    vec = {t: tf[t] * IDF.get(t, 0.0) for t in tf}
    return vec

DOC_VECS = [tfidf_vector(tokens) for tokens in DOC_TOKENS]


# Compute the cosine similarity for the search
def cosine(a: Dict[str, float], b: Dict[str, float]) -> float:

    if not a or not b:
        return 0.0

    dot = sum(a.get(k, 0.0) * b.get(k, 0.0) for k in set(a) | set(b))
    na = math.sqrt(sum(v*v for v in a.values()))
    nb = math.sqrt(sum(v*v for v in b.values()))
    return dot / (na * nb + 1e-12)


# Implement a search method based on the cosine similarity, which finds the documents with the highest similarity scores as the top-k search results.
def search_pets(query: str, k: int = 3, species: Optional[str] = None) -> List[Dict[str, Any]]:
    qvec = tfidf_vector(tokenize(query))
    scored = [(cosine(qvec, v), i) for i, v in enumerate(DOC_VECS)]
    scored.sort(reverse=True)

    filtered = []
    for score, idx in scored:
        d = CORPUS[idx]
        if species and d["species"].lower() != species.lower():
            continue
        filtered.append((score, idx))

    results = []
    for score, idx in filtered[:k]:
        d = CORPUS[idx].copy()
        d["score"] = float(score)
        results.append(d)
    return results

# Integrate the search method as a tool
def tool_search(query: str, k: int = 3, species: Optional[str] = None) -> Dict[str, Any]:
    hits = search_pets(query, k=k, species=species)
    return {
        "tool": "search",
        "query": query,
        "results": [
            {
                "id": h["id"],
                "name": h["name"],
                "species": h["species"],
                "breed": h["breed"],
                "age": h["age"],
                "sex": h["sex"],
                "location": h["location"],
                "score": h["score"],
                "snippet": h["description"][:200] + ("..." if len(h["description"]) > 200 else "")
            }
            for h in hits
        ],
    }

TOOLS = {
    "search": {
        "schema": {"query": "str", "k": "int? (default=3)", "species": "str?"},
        "fn": tool_search
    },
    "finish": {
        "schema": {"answer": "str"},
        "fn": lambda answer: {"tool": "finish", "answer": answer}
    }
}

In [72]:
import re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
LOAD_8BIT    = False
DTYPE        = torch.bfloat16 if torch.cuda.is_available() else torch.float32
MAX_NEW_TOKENS = 300
GENERATION_KWARGS = {}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            device_map="auto",
            torch_dtype=DTYPE,
            trust_remote_code=True,
            attn_implementation="eager",
            **({"load_in_8bit": True} if LOAD_8BIT else {})
        )

gen_cfg = GenerationConfig(
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=GENERATION_KWARGS.get("temperature", 0.1),
            do_sample=GENERATION_KWARGS.get("do_sample", True)
        )

T_PATTERN = re.compile(r"Thought:\s*(.+)")
A_PATTERN = re.compile(r"Action:\s*(.+)")

def _postprocess_to_two_lines(text: str) -> str:
    """
    Extract the first 'Thought:' and 'Action:' lines from the model output.
    If the model drifts, fall back to a conservative default Action.
    """
    text = text.split("\nObservation:")[0]
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]

    # Try to find explicit Thought/Action anywhere in the output
    thought = None
    action  = None
    for ln in lines:
        if thought is None:
            m = T_PATTERN.match(ln)
            if m:
                thought = m.group(1).strip()
                continue
        if action is None:
            m = A_PATTERN.match(ln)
            if m:
                action = m.group(1).strip()
                continue

    # Fallbacks if the model didn’t comply perfectly
    if thought is None:
        thought = "I should search for attributes described related to the requested animal."
    if action is None:
        action = 'search[query="(auto) refine the user question", k=3]'

    return f"Thought: {thought}\nAction: {action}"



def hf_llm(prompt: str) -> str:
    """
    Completes from your existing ReAct prompt and returns exactly two lines:
    'Thought: ...' and 'Action: ...'
    """
    # We add a strong instruction to the prompt to improve compliance with the format
    format_guard = (
        "\n\nIMPORTANT: Respond with EXACTLY two lines in this format:\n"
        "Thought: <one concise sentence>\n"
        "Action: <either search[query=\"...\"] or finish[answer=\"...\"]>\n"
        "Do NOT include Observation."
    )
    full_prompt = prompt + format_guard

    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, generation_config = gen_cfg)


    completion = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    return _postprocess_to_two_lines(completion)

LLM = hf_llm

In [74]:
from typing import Any, Dict, List, Optional, Tuple
import ast
import re
import textwrap
import csv, io

# ======= Helper functions =======
def convert_value(raw: str) -> Any:
    """
    Convert a raw string token into a Python type:
      - quoted strings -> str
      - numbers -> int or float
      - true/false -> bool
      - otherwise -> original string (stripped)
    Uses ast.literal_eval for safety (no code execution).
    """
    raw = raw.strip()
    # Normalize JSON-like booleans
    if raw.lower() == "true":  return True
    if raw.lower() == "false": return False
    try:
        # Handles "..." / '...' / 123 / 4.5
        return ast.literal_eval(raw)
    except Exception:
        # Fallback: unquoted, non-numeric tokens
        return raw.strip('"').strip("'")

def split_args(argstr: str) -> Dict[str, Any]:
    args: Dict[str, Any] = {}
    row = next(csv.reader(io.StringIO(argstr), delimiter=",", skipinitialspace=True, quotechar='"'), [])
    for field in row:
        field = field.strip()
        if not field:
            continue
        if "=" in field:
            key, val = field.split("=", 1)
            args[key.strip()] = convert_value(val)
        else:
            # bare flag -> True
            args[field] = True
    return args

# ====== Helper functions ======
def parse_action(line: str) -> Optional[Tuple[str, Dict[str, Any]]]:
    name = None; args = None

    line = line.strip()
    if not line.startswith("Action:"):
        return None

    rest = line[len("Action:"):].strip()

    match = re.match(r"^([a-zA-Z_][a-zA-Z0-9_]*)\s*\[(.*)\]\s*$", rest)
    if not match:
        return None

    name, argstr = match.groups()
    args = split_args(argstr)
    return name, args

# 2. We write a function that turn past steps into a readable history block for the prompt
def format_history(trajectory: List[Dict[str, str]]) -> str:
    """
    Each step in trajectory should have keys: 'thought', 'action', 'observation'.
    We render them in the canonical ReAct order for the next prompt.
    """
    lines: List[str] = []
    for step in trajectory:
        lines.append(f"Thought: {step['thought']}")
        lines.append(f"Action: {step['action']}")
        lines.append(f"Observation: {step['observation']}")
    return "\n".join(lines)


# 3. We will build the prompt shown to the model for the next step
SYSTEM_PREAMBLE = textwrap.dedent("""\
You are a helpful ReAct agent that helps users find adoptable pets.
You can use tools to search for pets based on their description, breed, age, species, location, and other attributes.

Available tools:
- search[query="<text>", species="<species>", k=<int>]  # searches the pet database and returns the top-k matching animals
  Default to k=3. Do not adjust unless absolutely necessary.
To finish, use: finish[answer="<final answer>"]

Follow the exact step format:
Thought: <your reasoning>
Action: <one of the tool calls above, or finish[...]>

Do not repeat a search with the exact same query if it has already been performed in this session.

### QUERY RULES
- Include **all descriptive words** from the user query in your search including personality traits.
  All information must go inside the `query` parameter.
  Do **NOT** add extra parameters to the search tools
  For example, "young cuddly calico cat Boston" must appear exactly in the search query.
- Always enforce species using the species parameter.
- Example search call:
  search[query="young cuddly calico Boston", species="cat", k=3]

### FINAL ANSWER FORMAT
- Only produce the final answer inside finish[answer="..."]
- Format each result using only attributes returned by the search results.
- Number the results:
  I have found <N> cats that match your description:  1. <name> — <age>, <breed> in <location>. <description> 2. <name> — <age>, <breed> in <location>. <description>
- Only use attributes returned by the search results. Do NOT infer missing data.
- Never merge animals into a single sentence or claim shared traits unless explicitly true.

IMPORTANT: Do not output the final answer before calling finish[...]. Only output your reasoning and Action steps before that.
""").strip()


def make_prompt(user_query: str, trajectory: List[Dict[str, str]]) -> str:
    """
    Build a prompt for the LLM including user query, history, and instructions.
    Ensures all descriptive words from the user query are preserved in the search query.
    """
    history_block = format_history(trajectory)

    prompt = (
        f"{SYSTEM_PREAMBLE}\n\n"
        f"User Question: {user_query}\n\n"
        f"{history_block}\n"
        f"Next step:\n"
        f"Thought:"
    )
    return prompt


In [70]:
from dataclasses import dataclass, field, asdict
from typing import Callable, Dict, List, Tuple, Optional, Any
import json, math, re, textwrap, random, os, sys
import math
from collections import Counter, defaultdict

@dataclass
class Step:
    thought: str
    action: str
    observation: str

@dataclass
class AgentConfig:
    max_steps: int = 3
    allow_tools: Tuple[str, ...] = ("search",)
    verbose: bool = True

class ReActAgent:
    def __init__(self, llm: Callable[[str], str], tools: Dict[str, Dict[str, Any]], config: AgentConfig | None=None):
        self.llm = llm
        self.tools = tools
        self.config = config or AgentConfig()
        self.trajectory: List[Step] = []

    def run(self, user_query: str) -> Dict[str, Any]:
        self.trajectory.clear()
        for step_idx in range(self.config.max_steps):
            # 1. At each step, format the prompt based on the make_prompt function and self.trajectory
            prompt = make_prompt(user_query, [asdict(s) for s in self.trajectory])

            # 2. Use self.llm to process the prompt
            out = self.llm(prompt)

            # Expect two lines: Thought:..., Action:...
            t_match = re.search(r"Thought:\s*(.*)", out)
            a_match = re.search(r"Action:\s*(.*)", out)
            thought = t_match.group(1).strip() if t_match else "(no thought)"
            action_line = a_match.group(0).strip() if a_match else 'Action: finish[answer="(no action)"]'

            #action_line = "Action: " + action_line


            # 3. Parse the action of the action line using the parse_action function
            parsed = parse_action(action_line)

            if not parsed:
                observation = "Invalid action format. Stopping."
                self.trajectory.append(Step(thought, action_line, observation))
                break
            name, args = parsed


            if name == "finish":
                observation = "done"
                self.trajectory.append(Step(thought, action_line, observation))
                break

            if name not in self.config.allow_tools or name not in self.tools:
                observation = f"Action '{name}' not allowed or not found."
                self.trajectory.append(Step(thought, action_line, observation))
                break

            # 4. Execute the action
            try:
                obs_payload = self.tools[name]["fn"](**args)
                observation = json.dumps(obs_payload, ensure_ascii=False)  # show structured obs
            except Exception as e:
                observation = f"Tool error: {e}"

            self.trajectory.append(Step(thought, action_line, observation))

        # Build final answer from last finish action if present
        final_answer = None
        for s in reversed(self.trajectory):
            if "finish[" in s.action:
                m = re.search(r'answer=["\'](.*?)["\']', s.action)
                if m:
                    final_answer = m.group(1)
                    break

        return {
            "question": user_query,
            "final_answer": final_answer,
            "steps": [asdict(s) for s in self.trajectory]
        }

In [84]:
agent = ReActAgent(llm=hf_llm, tools=TOOLS, config=None)

result = agent.run("I'm looking for a friendly dog in the Boston area.")

# Inspect the final answer and trajectory
print("Final answer:", result["final_answer"])
for step in result["steps"]:
    print(step["thought"])
    print(step["action"])
    print(step["observation"])
    print("---")


Final answer: I have found 2 dogs that match your description: 1. Buddy — 2 years old, male, in Rochester, NY. 2. Milo — 10 years old, female, in Cambridge, MA. They are friendly and intelligent, with a calm and friendly disposition.
I need to search for dogs in the Boston area with a friendly personality.
Action: search[query="friendly dog", species="dog", k=3]
{"tool": "search", "query": "friendly dog", "results": [{"id": "2012", "name": "Sadie", "species": "Dog", "breed": "Poodle", "age": "Senior", "sex": "Female", "location": "Poughkeepsie, NY", "score": 0.3230497952543346, "snippet": "Gentle, intelligent, with a calm and friendly disposition."}, {"id": "2002", "name": "Milo", "species": "Dog", "breed": "Beagle", "age": "Adult", "sex": "Male", "location": "Cambridge, MA", "score": 0.2852503453954425, "snippet": "Friendly beagle who enjoys long walks and sniffing everything."}, {"id": "2030", "name": "Buddy", "species": "Dog", "breed": "Jack Russell Terrier", "age": "Young", "sex": 